# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL and includes mass spectrometry analysis of extracellular vesicles from human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # Note: do NOT treat as a dict
print(f"Dataset title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets and their IDs.

We will list all available record sets, fields, and columns referenced by their Croissant `@id` values.

In [ ]:
# List record sets and their fields by @id
record_sets = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))  # handle possible naming
if not record_sets:
    # record_sets may not be directly present, so try loading them from dataset._record_sets (@id is always present)
    record_sets = list(dataset._record_sets.values())

record_set_ids = []
print('Available Record Sets:')
for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
    rs_name = getattr(rs, 'name', '')
    print(f"- @id: {rs_id} | Name: {rs_name}")
    record_set_ids.append(rs_id)
    # Fields
    fields = getattr(rs, 'fields', getattr(rs, 'field', getattr(rs, 'columns', [])))
    print('  Fields:')
    for field in fields:
        field_id = getattr(field, '@id', getattr(field, 'id', None))
        field_name = getattr(field, 'name', '')
        print(f"    * @id: {field_id}. Name: {field_name}")

## Record Example
Show an example row from each record set by their `@id` (using `dataset.records(record_set=...)`).

In [ ]:
for rs_id in record_set_ids:
    print(f"\nRecords for Record Set @id: {rs_id}")
    try:
        records = dataset.records(record_set=rs_id)
        for idx, r in enumerate(records):
            print({k: v for k, v in r.items()})
            if idx >= 1:
                break
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> Record sets, fields, and columns are referenced by their Croissant `@id` (see overview above).

We'll load all available record sets into pandas DataFrames for further exploration.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id} with {len(df)} records and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed loading {rs_id}: {e}")

# Peek into one record set with tabular data
if dataframes:
    preview_id = record_set_ids[0]  # use first available record set
    print(f"\nSample rows for Record Set @id: {preview_id}")
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field and perform some basic EDA operations:

- Filter rows by a threshold
- Normalize the numeric column
- Group by another field (if present)

Make sure to use the `@id` for field and grouping as referenced previously.

In [ ]:
# Choose record set and numeric field based on available columns
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

print(f"Available columns in {record_set_id}: {df.columns.tolist()}")

# Try to auto-detect a numeric field (float/int)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    print('No numeric field detected. Using the first one (if exists)')
    numeric_field_id = df.columns[0]

# Set threshold value for filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

# Filter rows and normalize
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try to group by another field
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < len(df) // 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped statistics by {group_field}:")
        display(grouped_df.head())
    else:
        print('No suitable group-by field detected in current record set.')
else:
    print('Selected field is not numeric, skipping EDA calculations.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the normalized values after filtering.

Feel free to adjust column names based on what you find in your dataset.

In [ ]:
import matplotlib.pyplot as plt

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    if filtered_df is not None and f'{numeric_field_id}_normalized' in filtered_df:
        plt.subplot(1, 2, 2)
        filtered_df[f'{numeric_field_id}_normalized'].hist(bins=30, color='orange')
        plt.title(f"Normalized {numeric_field_id} (Filtered)")
        plt.xlabel(f'{numeric_field_id}_normalized')

    plt.tight_layout()
    plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, programmatically discovering available record sets and columns. We applied basic data filtering, normalization, and grouped statistics using fields referenced by their Croissant `@id`. This approach ensures that all operations follow schema best practices, facilitating reproducible, standards-aligned scientific data analysis.